# Landing Gear / Terrain Collision Validation

Self-contained validation notebook for the SUAS landing gear model and terrain collision
model.  No Godot, no live sim, no terrain data download required.

**Scenario:** small UAS flying a *stabilized* approach to flat terrain at 0 m WGS84 elevation.
Starting from 4 m AGL on a 6° glideslope at a realistic approach speed (Vref ≈ 1.2 × stall), the
autopilot holds the flight-path angle (n_z = cos γ feed-forward + proportional FPA correction) all
the way to touchdown — no flare, so it arrives at the approach sink rate (a firm touchdown at
~1.4 × stall). (This replaces the earlier fixed-`n_z`=1
approach, which did not actually fly the descent — it leveled off and skimmed in at ~2.7 × stall.)

**What this tests:**
- Main gear touches first at ~0.23 m AGL, nose gear at ~0.21 m AGL
- Strut spring/damper forces should balance aircraft weight at steady state
- The aircraft settles onto the gear during roll-out (does not float on the wing)
- Body collider hard constraint prevents terrain penetration
- Contact forces reported through Python binding match expected values

Run all cells in order (`Kernel → Restart & Run All`).

In [ ]:
# ── Imports and path setup ────────────────────────────────────────────────────
import os, sys, json, math, copy
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML

# python/ directory is the parent of notebooks/
_PYTHON_DIR = Path('..').resolve()
if str(_PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(_PYTHON_DIR))

# The .pyd is compiled with MinGW and retains libwinpthread-1.dll as a dynamic
# dependency (brought in transitively via -lmingw32 from the SDL Conan package).
# Add the ucrt64 bin directory to the Windows DLL search path so the loader can
# find it without requiring ucrt64/bin on the system PATH.
if sys.platform == 'win32':
    os.add_dll_directory(r'C:\msys64\ucrt64\bin')

import liteaero_sim_py as sim
print(f'liteaero_sim_py loaded from {sim.__file__ if hasattr(sim, "__file__") else _PYTHON_DIR}')
print(f'Exported names: {[x for x in dir(sim) if not x.startswith("_")]}')

In [ ]:
# ── Aircraft configuration ─────────────────────────────────────────────────────
# Load the canonical SUAS config and patch the initial_state for the approach.

_REPO_ROOT  = _PYTHON_DIR.parent
_CONFIG_PATH = _REPO_ROOT / 'configs' / 'small_uas_ksba.json'

with open(_CONFIG_PATH) as f:
    base_cfg = json.load(f)

# Aircraft parameters from config
mass_kg  = base_cfg['inertia']['mass_kg']
S_ref_m2 = base_cfg['aircraft']['S_ref_m2']
cl_alpha = base_cfg['lift_curve']['cl_alpha']
cl_max   = base_cfg['lift_curve']['cl_max']
ar       = base_cfg['aircraft']['ar']
e        = base_cfg['aircraft']['e']
cd0      = base_cfg['aircraft']['cd0']

RHO_KGM3 = 1.225           # ISA sea-level air density
G_MPS2   = 9.80665
W_N      = mass_kg * G_MPS2
V_STALL_MPS = math.sqrt(2 * W_N / (RHO_KGM3 * S_ref_m2 * cl_max))

# ── Scenario parameters ────────────────────────────────────────────────────────
# A *flown* stabilized approach (not a fixed-load-factor controlled-flight-into-terrain):
# the aircraft holds the glideslope via flight-path-angle feedback (n_z = cos γ feed-forward
# plus a proportional FPA correction) ALL THE WAY to touchdown — no flare, so it arrives at the
# approach sink rate (a firm, realistic touchdown). Vref ≈ 1.2 × V_stall; from this short final
# the touchdown is ~1.4 × V_stall at the approach sink.
TERRAIN_ELEV_M   = 0.0     # flat terrain WGS84 elevation (m)
AGL_INIT_M       = 4.0     # initial altitude above terrain (m) — short final (low so it lands near Vref)
VREF_RATIO       = 1.20    # approach speed / stall speed (Vref)
V_APPROACH_MPS   = VREF_RATIO * V_STALL_MPS
GAMMA_DEG        = -6.0    # approach flight path angle (° — negative = descending)
HEADING_DEG      = 0.0     # heading (° true — 0 = north)
FPA_GAIN         = 2.0     # flight-path-angle-hold gain (Δn_z per rad of FPA error)
DT_S             = 0.01    # integration timestep (s) — 100 Hz
T_END_S          = 40.0    # simulation duration (s)

gamma_rad   = math.radians(GAMMA_DEG)
heading_rad = math.radians(HEADING_DEG)

q_inf_pa = 0.5 * RHO_KGM3 * V_APPROACH_MPS**2
qS       = q_inf_pa * S_ref_m2

# Trim lift coefficient and AoA at Vref on the glideslope (steady descent: n_z = cos γ)
CL_trim    = math.cos(gamma_rad) * W_N / qS
alpha_trim = CL_trim / cl_alpha   # rad (linear range)
pitch_init = alpha_trim + gamma_rad

# NED velocities (heading north, descending)
vN = V_APPROACH_MPS * math.cos(abs(gamma_rad)) * math.cos(heading_rad)
vE = V_APPROACH_MPS * math.cos(abs(gamma_rad)) * math.sin(heading_rad)
vD = V_APPROACH_MPS * math.sin(abs(gamma_rad))   # positive = downward in NED

# Patch initial_state — use equator/prime-meridian so ENU≈flat-Earth holds
approach_cfg = copy.deepcopy(base_cfg)
approach_cfg['initial_state'] = {
    'latitude_rad':       0.0,
    'longitude_rad':      0.0,
    'altitude_m':         TERRAIN_ELEV_M + AGL_INIT_M,
    'velocity_north_mps': vN,
    'velocity_east_mps':  vE,
    'velocity_down_mps':  vD,
    'wind_north_mps':     0.0,
    'wind_east_mps':      0.0,
    'wind_down_mps':      0.0,
    'pitch_rad':          pitch_init,
    'roll_rad':           0.0,
    'heading_rad':        heading_rad,
}

print(f'Aircraft mass       = {mass_kg:.1f} kg   weight = {W_N:.1f} N   V_stall = {V_STALL_MPS:.2f} m/s')
print(f'Approach Vref       = {V_APPROACH_MPS:.2f} m/s  ({VREF_RATIO:.2f}x stall)   gamma = {GAMMA_DEG:.1f} deg')
print(f'Trim CL             = {CL_trim:.3f}   alpha_trim = {math.degrees(alpha_trim):.2f} deg')
print(f'Initial pitch       = {math.degrees(pitch_init):.2f} deg')
print(f'NED velocity        = [{vN:.2f}, {vE:.2f}, {vD:.2f}] m/s')
print(f'Initial altitude    = {AGL_INIT_M:.1f} m AGL   (no flare — firm touchdown at the approach sink)')

In [ ]:
# ── Gear geometry (from config, for visualization reference) ──────────────────
gear_params = approach_cfg['landing_gear']['wheel_units']
GEAR_NAMES  = ['Nose', 'L-Main', 'R-Main']
GEAR_COLORS = ['tab:blue', 'tab:orange', 'tab:green']

# At zero deflection, contact patch altitude offset below CG (level aircraft):
#   offset = attach.z + tyre_radius   (body +Z = down)
for i, (g, name) in enumerate(zip(gear_params, GEAR_NAMES)):
    attach = g['attach_point_body_m']
    r      = g['tyre_radius_m']
    zmax   = g['travel_max_m']
    offset = attach[2] + r
    print(f'{name:8s}  attach={attach}  tyre_r={r:.2f} m  '
          f'travel_max={zmax:.2f} m  '
          f'first-contact AGL ≈ {offset:.3f} m  '
          f'k={g["spring_stiffness_npm"]:.0f} N/m')

# Static deflection at steady state (main gear pair carries ~all weight at low speed)
k_main   = gear_params[1]['spring_stiffness_npm']
delta_static = W_N / (2 * k_main)
print(f'\nExpected static main-gear deflection (each) ≈ {delta_static*1000:.1f} mm'
      f'  (W / (2·k) = {W_N:.1f} / {2*k_main:.0f})')

In [ ]:
# ── Simulation helper ──────────────────────────────────────────────────────────
def _run_sim(cfg, label=''):
    """Fly a stabilized FPA-hold approach + flare to touchdown; return result arrays."""
    terrain_local = sim.FlatTerrain(TERRAIN_ELEV_M)
    ac_local = sim.Aircraft(json.dumps(cfg), dt_s=DT_S)
    ac_local.set_terrain(terrain_local)

    cmd_local = sim.AircraftCommand()
    cmd_local.n_y               = 0.0
    cmd_local.roll_rate_wind_rps = 0.0
    cmd_local.throttle_nd       = 0.0          # idle approach

    N_local = int(T_END_S / DT_S)
    r = dict(
        t_s          = np.zeros(N_local),
        alt_m        = np.zeros(N_local),
        agl_m        = np.zeros(N_local),
        vel_ned      = np.zeros((N_local, 3)),
        vel_body     = np.zeros((N_local, 3)),
        accel_ned    = np.zeros((N_local, 3)),
        pitch_rad    = np.zeros(N_local),
        roll_rad_    = np.zeros(N_local),
        heading_rad_ = np.zeros(N_local),
        alpha_rad    = np.zeros(N_local),
        beta_rad     = np.zeros(N_local),
        p_rad_s      = np.zeros(N_local),
        q_rad_s      = np.zeros(N_local),
        r_rad_s      = np.zeros(N_local),
        airspeed     = np.zeros(N_local),
        cl_eff       = np.zeros(N_local),
        n_z_cmd      = np.zeros(N_local),
        cf_force     = np.zeros((N_local, 3)),
        cf_moment    = np.zeros((N_local, 3)),
        wow          = np.zeros(N_local, dtype=bool),
        strut_defl   = np.zeros((N_local, 3)),
        strut_rate   = np.zeros((N_local, 3)),
        wheel_spd    = np.zeros((N_local, 3)),
    )

    gamma_target = abs(gamma_rad)                  # descent-positive glideslope target

    for i in range(N_local):
        s  = ac_local.state()
        cf = ac_local.contact_forces()
        gs = ac_local.gear_strut_states()

        # ── Approach guidance: hold the glideslope to touchdown (no flare); hold on the ground ──
        if cf.weight_on_wheels:
            cmd_local.n_z = 1.0
        else:
            v_h = math.hypot(s.velocity_north_mps, s.velocity_east_mps)
            gamma_cur = math.atan2(s.velocity_down_mps, max(v_h, 0.1))
            cmd_local.n_z = max(0.2, min(1.8, math.cos(gamma_target) + FPA_GAIN * (gamma_cur - gamma_target)))

        r['t_s'][i]         = s.time_s
        r['alt_m'][i]       = s.altitude_m
        r['agl_m'][i]       = ac_local.agl_m()
        r['vel_ned'][i]     = [s.velocity_north_mps, s.velocity_east_mps, s.velocity_down_mps]
        r['vel_body'][i]    = s.velocity_body_mps
        r['accel_ned'][i]   = s.acceleration_ned_mps2
        r['pitch_rad'][i]   = s.pitch_rad
        r['roll_rad_'][i]   = s.roll_rad
        r['heading_rad_'][i]= s.heading_rad
        r['alpha_rad'][i]   = s.alpha_rad
        r['beta_rad'][i]    = s.beta_rad
        r['p_rad_s'][i]     = s.rates_body_rad_s[0]
        r['q_rad_s'][i]     = s.rates_body_rad_s[1]
        r['r_rad_s'][i]     = s.rates_body_rad_s[2]
        r['airspeed'][i]    = s.airspeed_m_s
        r['cl_eff'][i]      = ac_local.cl_eff
        r['n_z_cmd'][i]     = cmd_local.n_z
        r['cf_force'][i]    = cf.force_body_n
        r['cf_moment'][i]   = cf.moment_body_nm
        r['wow'][i]         = cf.weight_on_wheels
        if gs:
            for j, g in enumerate(gs):
                r['strut_defl'][i, j] = g.deflection_m
                r['strut_rate'][i, j] = g.deflection_rate_mps
                r['wheel_spd'][i, j]  = g.wheel_speed_rps

        ac_local.step(cmd_local, dt_s=DT_S, rho_kgm3=RHO_KGM3)

    td_idx = np.argmax(r['wow']) if np.any(r['wow']) else None
    r['td_t'] = r['t_s'][td_idx] if td_idx is not None else None
    r['label'] = label
    return r

# ── Run: with landing gear (primary) ──────────────────────────────────────────
results_gear    = _run_sim(approach_cfg,         label='With gear')

# ── Run: body-collider only (no landing gear) ─────────────────────────────────
_cfg_no_gear = copy.deepcopy(approach_cfg)
del _cfg_no_gear['landing_gear']
results_no_gear = _run_sim(_cfg_no_gear, label='No gear (body collider only)')

# ── Unpack primary (with-gear) results into module-level names used by cells below
t_s          = results_gear['t_s']
alt_m        = results_gear['alt_m']
agl_m        = results_gear['agl_m']
vel_ned      = results_gear['vel_ned']
vel_body     = results_gear['vel_body']
accel_ned    = results_gear['accel_ned']
pitch_rad    = results_gear['pitch_rad']
roll_rad_    = results_gear['roll_rad_']
heading_rad_ = results_gear['heading_rad_']
alpha_rad    = results_gear['alpha_rad']
beta_rad     = results_gear['beta_rad']
p_rad_s      = results_gear['p_rad_s']
q_rad_s      = results_gear['q_rad_s']
r_rad_s      = results_gear['r_rad_s']
airspeed     = results_gear['airspeed']
cl_eff       = results_gear['cl_eff']
cf_force     = results_gear['cf_force']
cf_moment    = results_gear['cf_moment']
wow          = results_gear['wow']
strut_defl   = results_gear['strut_defl']
strut_rate   = results_gear['strut_rate']
wheel_spd    = results_gear['wheel_spd']
td_t_main    = results_gear['td_t']

N = len(t_s)

# Derived: position integrated from NED velocity (relative to start)
north_m = np.cumsum(vel_ned[:, 0]) * DT_S
east_m  = np.cumsum(vel_ned[:, 1]) * DT_S

# Derived: aerodynamic forces. Lift uses the model's *effective* CL (cl_eff) — the actual
# aerodynamic CL applied — NOT a lift-curve reconstruction from the body-attitude alpha (which
# includes the gear-induced nose-down deviation on the ground and would imply phantom downforce).
k_ind   = 1.0 / (math.pi * ar * e)
CL_nom  = alpha_rad * cl_alpha          # nominal lift-curve value from body alpha (reference only)
CD      = cd0 + k_ind * cl_eff**2
q_dyn   = 0.5 * RHO_KGM3 * airspeed**2
L_N     = q_dyn * S_ref_m2 * cl_eff     # actual aero lift
D_N     = q_dyn * S_ref_m2 * CD

print(f'With-gear run:    {N} steps   first WoW at t = {td_t_main:.3f} s' if td_t_main else f'With-gear run: {N} steps   no WoW detected')
ng_td = results_no_gear['td_t']
print(f'No-gear run:      {N} steps   first WoW at t = {ng_td:.3f} s' if ng_td else f'No-gear run: {N} steps   no WoW detected')
if td_t_main is not None:
    td_i = int(np.argmax(wow))
    print(f'Touchdown speed   = {airspeed[td_i]:.2f} m/s  ({airspeed[td_i]/V_STALL_MPS:.2f}x stall)   '
          f'sink = {vel_ned[td_i, 2]:.2f} m/s')

In [ ]:
# ── Figure 0: Gear vs No-gear comparison ──────────────────────────────────────
_COLORS = {'With gear': 'tab:blue', 'No gear (body collider only)': 'tab:orange'}

fig0, axes0 = plt.subplots(2, 2, figsize=(13, 8))
fig0.suptitle('Gear vs No-gear — Key Metrics Comparison', fontsize=12, fontweight='bold')

for rr in [results_gear, results_no_gear]:
    col   = _COLORS[rr['label']]
    lbl   = rr['label']
    td_t  = rr['td_t']
    t     = rr['t_s']
    agl   = rr['agl_m']
    fz    = -rr['cf_force'][:, 2]   # upward contact force
    pitch = np.degrees(rr['pitch_rad'])
    wow_f = rr['wow'].astype(float)

    # AGL
    ax = axes0[0, 0]
    ax.plot(t, agl, color=col, label=lbl)

    # Contact force Fz vs weight
    ax = axes0[0, 1]
    ax.plot(t, fz, color=col, label=lbl)

    # Pitch angle
    ax = axes0[1, 0]
    ax.plot(t, pitch, color=col, label=lbl)

    # WoW
    ax = axes0[1, 1]
    ax.plot(t, wow_f, color=col, label=lbl, lw=1.2)
    if td_t is not None:
        for ax_ in axes0.flat:
            ax_.axvline(td_t, color=col, lw=0.8, ls=':', alpha=0.6)

# Decorate
ax = axes0[0, 0]
ax.axhline(0.0, color='saddlebrown', lw=1.5)
ax.set_xlabel('Time (s)'); ax.set_ylabel('AGL (m)')
ax.set_title('Height Above Ground (CG)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes0[0, 1]
ax.axhline(W_N, color='k', lw=1.0, ls='--', label=f'Weight {W_N:.1f} N')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Force (N)')
ax.set_title('Contact Force −Fz (upward reaction)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes0[1, 0]
ax.set_xlabel('Time (s)'); ax.set_ylabel('Pitch (°)')
ax.set_title('Pitch Angle'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes0[1, 1]
ax.set_ylim(-0.1, 1.3)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Weight-on-Wheels')
ax.set_title('Weight-on-Wheels Flag'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig0.tight_layout()
plt.show()

# ── Steady-state summary: both scenarios ──────────────────────────────────────
print()
for rr in [results_gear, results_no_gear]:
    lbl     = rr['label']
    t       = rr['t_s']
    ss_mask = (t > T_END_S - 2.0) & rr['wow']
    print(f'── {lbl} ──')
    if ss_mask.sum() < 5:
        print('  WARNING: < 5 WoW samples in last 2 s — aircraft not settled on ground.')
    else:
        mean_Fz   = rr['cf_force'][ss_mask, 2].mean()
        mean_agl  = rr['agl_m'][ss_mask].mean()
        fz_err    = abs((mean_Fz + W_N) / W_N) * 100
        mean_pitch = np.degrees(rr['pitch_rad'][ss_mask].mean())
        print(f'  Mean contact Fz  = {mean_Fz:.1f} N  (expected ≈ {-W_N:.1f} N)  error = {fz_err:.1f}%')
        print(f'  Mean AGL         = {mean_agl:.4f} m')
        print(f'  Mean pitch       = {mean_pitch:.2f}°')
        if fz_err > 5.0:
            print(f'  ⚠  Fz error {fz_err:.1f}% > 5%')
        if mean_agl < -0.01:
            print(f'  ⚠  Negative AGL — terrain penetration detected')
    print()

In [ ]:
# ── Helper: annotate touchdown on an axis ─────────────────────────────────────
def _annotate_td(ax, td_t, label='T/D', color='red', alpha=0.7):
    if td_t is None:
        return
    ax.axvline(td_t, color=color, lw=1.0, ls='--', alpha=alpha)
    ylim = ax.get_ylim()
    ax.text(td_t + 0.1, ylim[0] + 0.05*(ylim[1]-ylim[0]),
            label, color=color, fontsize=7, alpha=alpha)

In [ ]:
# ── Figure 1: Position and kinematics ─────────────────────────────────────────
fig1, axes1 = plt.subplots(3, 2, figsize=(13, 10))
fig1.suptitle('SUAS Landing Approach — Position and Kinematics', fontsize=12, fontweight='bold')

# Altitude
ax = axes1[0, 0]
ax.plot(t_s, alt_m, 'k')
ax.axhline(TERRAIN_ELEV_M, color='saddlebrown', lw=1.5, ls='-', label=f'terrain {TERRAIN_ELEV_M:.0f} m')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Altitude WGS84 (m)')
ax.set_title('CG Altitude'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# AGL
ax = axes1[0, 1]
ax.plot(t_s, agl_m, 'k')
ax.axhline(0.0, color='saddlebrown', lw=1.5)
gear_offsets = [p['attach_point_body_m'][2] + p['tyre_radius_m'] for p in gear_params]
for offset, name, col in zip(gear_offsets, GEAR_NAMES, GEAR_COLORS):
    ax.axhline(offset, color=col, lw=1, ls=':', alpha=0.7, label=f'{name} contact ≈{offset:.3f} m')
ax.set_xlabel('Time (s)'); ax.set_ylabel('AGL (m)')
ax.set_title('Height Above Ground Level (CG)')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# NED velocity
ax = axes1[1, 0]
ax.plot(t_s, vel_ned[:, 0], label='v_N')
ax.plot(t_s, vel_ned[:, 1], label='v_E')
ax.plot(t_s, vel_ned[:, 2], label='v_D (+ down)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Velocity (m/s)')
ax.set_title('NED Velocity'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Body velocity
ax = axes1[1, 1]
ax.plot(t_s, vel_body[:, 0], label='u (fwd)')
ax.plot(t_s, vel_body[:, 1], label='v (right)')
ax.plot(t_s, vel_body[:, 2], label='w (down)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Velocity (m/s)')
ax.set_title('Body-Frame Velocity'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Euler angles
ax = axes1[2, 0]
ax.plot(t_s, np.degrees(pitch_rad),    label='pitch θ')
ax.plot(t_s, np.degrees(roll_rad_),    label='roll φ')
ax.plot(t_s, np.degrees(alpha_rad),    label='α AoA', ls='--')
ax.plot(t_s, np.degrees(beta_rad),     label='β sideslip', ls='--')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Angle (°)')
ax.set_title('Euler Angles and Aerodynamic Angles'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Angular rates
ax = axes1[2, 1]
ax.plot(t_s, np.degrees(p_rad_s), label='p (roll)')
ax.plot(t_s, np.degrees(q_rad_s), label='q (pitch)')
ax.plot(t_s, np.degrees(r_rad_s), label='r (yaw)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Rate (°/s)')
ax.set_title('Body Angular Rates'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

fig1.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Ground contact ───────────────────────────────────────────────────
fig2, axes2 = plt.subplots(3, 2, figsize=(13, 10))
fig2.suptitle('SUAS Landing Approach — Ground Contact', fontsize=12, fontweight='bold')

# WOW flag
ax = axes2[0, 0]
ax.fill_between(t_s, wow.astype(float), alpha=0.4, color='tab:red', label='WoW')
ax.plot(t_s, wow.astype(float), 'tab:red', lw=0.8)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Weight-on-Wheels')
ax.set_ylim(-0.1, 1.3); ax.set_title('Weight-on-Wheels Flag'); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Strut deflections
ax = axes2[0, 1]
for j, (name, col) in enumerate(zip(GEAR_NAMES, GEAR_COLORS)):
    ax.plot(t_s, strut_defl[:, j]*1000, color=col, label=name)
    ax.axhline(delta_static*1000, color=col, lw=0.8, ls=':', alpha=0.6)
ax.text(t_s[-1]*0.02, delta_static*1000 + 0.3, f'static δ_main={delta_static*1000:.1f} mm', fontsize=7, color='gray')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Deflection (mm)')
ax.set_title('Strut Deflection (compression)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Strut rates
ax = axes2[1, 0]
for j, (name, col) in enumerate(zip(GEAR_NAMES, GEAR_COLORS)):
    ax.plot(t_s, strut_rate[:, j]*1000, color=col, label=name)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Rate (mm/s)')
ax.set_title('Strut Compression Rate'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Wheel speed
ax = axes2[1, 1]
for j, (name, col) in enumerate(zip(GEAR_NAMES, GEAR_COLORS)):
    ax.plot(t_s, wheel_spd[:, j], color=col, label=name)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Wheel speed (rad/s)')
ax.set_title('Wheel Angular Speed'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Contact force body-Z (normal) — should approach W_N after settling
ax = axes2[2, 0]
ax.plot(t_s, -cf_force[:, 2], label='Fz body (up = −Fz_body)')
ax.axhline(W_N, color='k', lw=1.0, ls='--', label=f'Weight {W_N:.1f} N')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Force (N)')
ax.set_title('Contact Normal Force vs Aircraft Weight'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Contact force body-X and Y (longitudinal and lateral friction)
ax = axes2[2, 1]
ax.plot(t_s, cf_force[:, 0], label='Fx (fwd)')
ax.plot(t_s, cf_force[:, 1], label='Fy (right)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Force (N)')
ax.set_title('Contact Friction Forces (body frame)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

fig2.tight_layout()
plt.show()

In [ ]:
# ── Figure 3: Contact moments and acceleration ─────────────────────────────────
fig3, axes3 = plt.subplots(2, 2, figsize=(13, 7))
fig3.suptitle('SUAS Landing Approach — Moments and Acceleration', fontsize=12, fontweight='bold')

# Contact moments body
ax = axes3[0, 0]
ax.plot(t_s, cf_moment[:, 0], label='Mx (roll)')
ax.plot(t_s, cf_moment[:, 1], label='My (pitch)')
ax.plot(t_s, cf_moment[:, 2], label='Mz (yaw)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Moment (N·m)')
ax.set_title('Contact Moments (body frame)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# NED acceleration
ax = axes3[0, 1]
ax.plot(t_s, accel_ned[:, 0], label='aN')
ax.plot(t_s, accel_ned[:, 1], label='aE')
ax.plot(t_s, accel_ned[:, 2], label='aD (+down)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Acceleration (m/s²)')
ax.set_title('NED Acceleration'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Aerodynamic coefficients. cl_eff is the model's actual applied CL; CL_nom is the lift-curve
# value at the body-attitude alpha (drops below cl_eff on the ground, where the gear pitches the
# body nose-down — applying CL_nom there would imply phantom downforce the model does not produce).
ax = axes3[1, 0]
ax.plot(t_s, cl_eff, label='C_L (effective, applied)')
ax.plot(t_s, CL_nom, label='C_L (nominal from body α)', ls=':')
ax.plot(t_s, CD, label='C_D')
ax.plot(t_s, cl_eff/np.maximum(CD, 1e-6), label='L/D', ls='--')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Coefficient / ratio')
ax.set_title('Lift/Drag Coefficients'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

# Lift and Drag forces (lift from the effective CL — the actual aerodynamic force)
ax = axes3[1, 1]
ax.plot(t_s, L_N, label='Lift L (N)')
ax.plot(t_s, D_N, label='Drag D (N)')
ax.axhline(W_N, color='k', lw=0.8, ls='--', label=f'Weight {W_N:.1f} N')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Force (N)')
ax.set_title('Aerodynamic Forces (lift from effective CL)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
_annotate_td(ax, td_t_main)

fig3.tight_layout()
plt.show()

In [ ]:
# ── Wireframe animation helpers ────────────────────────────────────────────────

def box_edges_body(center, half):
    """Return the 12 edges of an axis-aligned box as list of (pt1, pt2) in body frame."""
    c, h = np.array(center), np.array(half)
    signs = [(-1,-1,-1),(-1,-1,1),(-1,1,-1),(-1,1,1),(1,-1,-1),(1,-1,1),(1,1,-1),(1,1,1)]
    corners = np.array([c + np.array([s[0]*h[0], s[1]*h[1], s[2]*h[2]]) for s in signs])
    edges = []
    for i in range(8):
        for j in range(i+1, 8):
            if np.sum(np.abs(corners[i] - corners[j]) > 1e-6) == 1:
                edges.append((corners[i], corners[j]))
    return edges, corners

def euler_to_Rnb(heading, pitch, roll):
    """ZYX Euler angles → body-to-NED rotation matrix (float64)."""
    ch, sh = math.cos(heading), math.sin(heading)
    cp, sp = math.cos(pitch),   math.sin(pitch)
    cr, sr = math.cos(roll),    math.sin(roll)
    return np.array([
        [ch*cp,  ch*sp*sr - sh*cr,  ch*sp*cr + sh*sr],
        [sh*cp,  sh*sp*sr + ch*cr,  sh*sp*cr - ch*sr],
        [-sp,    cp*sr,             cp*cr           ]
    ])

def body_to_enu(pts_body, R_nb, cg_enu):
    """Transform body-frame points (N×3) to ENU world frame.
    ENU = [E, N, U] = [NED_y, NED_x, -NED_z]"""
    pts_ned = (R_nb @ pts_body.T).T
    pts_enu = np.column_stack([pts_ned[:, 1], pts_ned[:, 0], -pts_ned[:, 2]])
    return pts_enu + cg_enu

# Aircraft geometry in body frame (+X forward, +Y right, +Z down)
VOL_DEFS = [
    ('fuselage',  approach_cfg['body_collider']['volumes'][0]['center_offset_body_m'],
                  approach_cfg['body_collider']['volumes'][0]['half_extents_body_m'], 'gray'),
    ('wing',      approach_cfg['body_collider']['volumes'][1]['center_offset_body_m'],
                  approach_cfg['body_collider']['volumes'][1]['half_extents_body_m'], 'steelblue'),
    ('h_tail',    approach_cfg['body_collider']['volumes'][2]['center_offset_body_m'],
                  approach_cfg['body_collider']['volumes'][2]['half_extents_body_m'], 'steelblue'),
]

# Precompute all box edge data in body frame
VOL_EDGES = [(box_edges_body(c, h), col) for (_, c, h, col) in VOL_DEFS]

# Gear attach and tyre data
GEAR_ATTACH  = np.array([g['attach_point_body_m'] for g in gear_params])   # (3,3)
GEAR_AXIS    = np.array([g['travel_axis_body']     for g in gear_params])   # (3,3)
GEAR_RADIUS  = np.array([g['tyre_radius_m']        for g in gear_params])   # (3,)

def gear_body_pts(deflections):
    """Return upper (attach) and lower (contact patch) body-frame points for each strut."""
    upper = GEAR_ATTACH.copy()                                    # (3,3)
    lower = GEAR_ATTACH + (deflections[:, None] + GEAR_RADIUS[:, None]) * GEAR_AXIS  # (3,3)
    return upper, lower

# ENU trajectory
cg_enu = np.column_stack([east_m, north_m, alt_m - TERRAIN_ELEV_M])  # altitude above terrain

print('Wireframe helpers ready.')
print(f'Trajectory: {cg_enu.shape[0]} frames, N range [{cg_enu[:,1].min():.1f}, {cg_enu[:,1].max():.1f}] m')

In [ ]:
# ── Wireframe animation — 4-view ──────────────────────────────────────────────
#
# Side view  (XZ): North vs Up — aircraft profile
# Top view   (XY): East vs North — ground track
# Rear view  (YZ): East vs Up — cross-section looking forward
# Isometric  (3D): East / North / Up
#
# Animation frame rate is decimated for display speed.

ANIM_DECIMATE = 10   # show every Nth simulation step
GROUND_HALF   = 5.0  # half-width of ground patch in world coords (m)

anim_idx = np.arange(0, N, ANIM_DECIMATE)
print(f'Animation: {len(anim_idx)} frames  (every {ANIM_DECIMATE} steps = {ANIM_DECIMATE*DT_S*1000:.0f} ms/frame)')

fig_anim = plt.figure(figsize=(14, 11), layout='constrained')
ax_side = fig_anim.add_subplot(2, 2, 1)              # North vs Up
ax_top  = fig_anim.add_subplot(2, 2, 2)              # East  vs North
ax_rear = fig_anim.add_subplot(2, 2, 3)              # East  vs Up
ax_3d   = fig_anim.add_subplot(2, 2, 4, projection='3d')  # 3D
fig_anim.suptitle('SUAS Landing Approach — 4-View Wireframe', fontsize=11, fontweight='bold')

# Axis labels
ax_side.set_xlabel('North (m)'); ax_side.set_ylabel('Up (m)');   ax_side.set_title('Side View')
ax_top.set_xlabel('East (m)');  ax_top.set_ylabel('North (m)'); ax_top.set_title('Top View')
ax_rear.set_xlabel('East (m)'); ax_rear.set_ylabel('Up (m)');   ax_rear.set_title('Rear View')
ax_3d.set_xlabel('East');       ax_3d.set_ylabel('North');      ax_3d.set_zlabel('Up'); ax_3d.set_title('Isometric')

# Ground plane patches
# Side: horizontal line at Up=0
N_range  = [north_m.min() - 1, north_m.max() + 1]
E_range  = [east_m.min()  - 1, east_m.max()  + 1]
Up_range = [-0.5, AGL_INIT_M + 1]

ax_side.axhline(0, color='saddlebrown', lw=2)
ax_side.fill_between(N_range, [0, 0], [-0.5, -0.5], color='saddlebrown', alpha=0.2)
ax_side.set_xlim(N_range); ax_side.set_ylim(Up_range)

ax_top.axhline(0, color='saddlebrown', lw=0.5, ls=':')
ax_top.set_xlim(E_range); ax_top.set_ylim(N_range)

ax_rear.axhline(0, color='saddlebrown', lw=2)
ax_rear.fill_between(E_range, [0, 0], [-0.5, -0.5], color='saddlebrown', alpha=0.2)
ax_rear.set_xlim(E_range); ax_rear.set_ylim(Up_range)

# Trajectory ghost in grey
ax_side.plot(cg_enu[:, 1], cg_enu[:, 2], 'lightgray', lw=0.8, zorder=1)
ax_top.plot( cg_enu[:, 0], cg_enu[:, 1], 'lightgray', lw=0.8, zorder=1)
ax_rear.plot(cg_enu[:, 0], cg_enu[:, 2], 'lightgray', lw=0.8, zorder=1)

for ax in [ax_side, ax_top, ax_rear]:
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax_3d.set_box_aspect([1, 3, 1])

# ── Artists (to be updated each frame) ────────────────────────────────────────
# CG marker
cg_side, = ax_side.plot([], [], 'ko', ms=4, zorder=5)
cg_top,  = ax_top.plot([],  [], 'ko', ms=4, zorder=5)
cg_rear, = ax_rear.plot([], [], 'ko', ms=4, zorder=5)
cg_3d    = ax_3d.plot([],   [], [],  'ko', ms=4)[0]

# Time label
time_text = ax_side.text(0.02, 0.96, '', transform=ax_side.transAxes, fontsize=8, va='top')

# Body volume edge line collections (one set per volume, per view)
def _make_body_lines(ax, n_vols, col3d=False):
    lines = []
    for (_, c, h, col), ((edges, corners), _) in zip(VOL_DEFS, VOL_EDGES):
        vol_lines = []
        for _ in edges:
            if col3d:
                ln, = ax.plot([], [], [], color=col, lw=0.8)
            else:
                ln, = ax.plot([], [], color=col, lw=0.8)
            vol_lines.append(ln)
        lines.append(vol_lines)
    return lines

body_lines_side = _make_body_lines(ax_side, len(VOL_DEFS))
body_lines_top  = _make_body_lines(ax_top,  len(VOL_DEFS))
body_lines_rear = _make_body_lines(ax_rear, len(VOL_DEFS))
body_lines_3d   = _make_body_lines(ax_3d,   len(VOL_DEFS), col3d=True)

# Gear strut lines and wheel-circle ellipses
_strut_side  = [ax_side.plot([], [], color=c, lw=1.5)[0] for c in GEAR_COLORS]
_strut_top   = [ax_top.plot([],  [], color=c, lw=1.5)[0] for c in GEAR_COLORS]
_strut_rear  = [ax_rear.plot([], [], color=c, lw=1.5)[0] for c in GEAR_COLORS]
_strut_3d    = [ax_3d.plot([],   [], [], color=c, lw=1.5)[0] for c in GEAR_COLORS]
_contact_dot_side = [ax_side.plot([], [], 'o', color=c, ms=3)[0] for c in GEAR_COLORS]
_contact_dot_top  = [ax_top.plot([],  [], 'o', color=c, ms=3)[0] for c in GEAR_COLORS]
_contact_dot_rear = [ax_rear.plot([], [], 'o', color=c, ms=3)[0] for c in GEAR_COLORS]

# ── Update function ───────────────────────────────────────────────────────────
def _update(frame_num):
    k = anim_idx[frame_num]
    R_nb = euler_to_Rnb(heading_rad_[k], pitch_rad[k], roll_rad_[k])

    cg_e  = cg_enu[k, 0]
    cg_n  = cg_enu[k, 1]
    cg_u  = cg_enu[k, 2]
    cg_pt = cg_enu[k:k+1, :]

    cg_side.set_data([cg_n], [cg_u])
    cg_top.set_data( [cg_e], [cg_n])
    cg_rear.set_data([cg_e], [cg_u])
    cg_3d.set_data_3d([cg_e], [cg_n], [cg_u])

    time_text.set_text(f't = {t_s[k]:.2f} s  AGL = {agl_m[k]:.2f} m')

    # Body wireframe
    for vi, ((edges, corners), col) in enumerate(VOL_EDGES):
        pts_enu = body_to_enu(corners, R_nb, cg_pt)  # (8,3)
        # Rebuild edge connectivity in ENU
        edges_enu = []
        for ei, (p1, p2) in enumerate(edges):
            p1_enu = body_to_enu(p1[np.newaxis], R_nb, cg_pt)[0]
            p2_enu = body_to_enu(p2[np.newaxis], R_nb, cg_pt)[0]
            body_lines_side[vi][ei].set_data([p1_enu[1], p2_enu[1]], [p1_enu[2], p2_enu[2]])
            body_lines_top[vi][ei].set_data( [p1_enu[0], p2_enu[0]], [p1_enu[1], p2_enu[1]])
            body_lines_rear[vi][ei].set_data([p1_enu[0], p2_enu[0]], [p1_enu[2], p2_enu[2]])
            body_lines_3d[vi][ei].set_data_3d([p1_enu[0], p2_enu[0]],
                                              [p1_enu[1], p2_enu[1]],
                                              [p1_enu[2], p2_enu[2]])

    # Landing gear struts
    uppers, lowers = gear_body_pts(strut_defl[k])
    for gi in range(3):
        u_enu = body_to_enu(uppers[gi:gi+1], R_nb, cg_pt)[0]
        l_enu = body_to_enu(lowers[gi:gi+1], R_nb, cg_pt)[0]
        _strut_side[gi].set_data([u_enu[1], l_enu[1]], [u_enu[2], l_enu[2]])
        _strut_top[gi].set_data( [u_enu[0], l_enu[0]], [u_enu[1], l_enu[1]])
        _strut_rear[gi].set_data([u_enu[0], l_enu[0]], [u_enu[2], l_enu[2]])
        _strut_3d[gi].set_data_3d([u_enu[0], l_enu[0]],
                                   [u_enu[1], l_enu[1]],
                                   [u_enu[2], l_enu[2]])
        # contact dot (bottom of strut)
        _contact_dot_side[gi].set_data([l_enu[1]], [l_enu[2]])
        _contact_dot_top[gi].set_data( [l_enu[0]], [l_enu[1]])
        _contact_dot_rear[gi].set_data([l_enu[0]], [l_enu[2]])

    # 3D axes range (follow aircraft)
    ax_3d.set_xlim(cg_e - 2, cg_e + 2)
    ax_3d.set_ylim(cg_n - 3, cg_n + 3)
    ax_3d.set_zlim(-0.2, 3.0)

    return (cg_side, cg_top, cg_rear, cg_3d, time_text,
            *[ln for vol in body_lines_side for ln in vol],
            *[ln for vol in body_lines_top  for ln in vol],
            *[ln for vol in body_lines_rear for ln in vol],
            *[ln for vol in body_lines_3d   for ln in vol],
            *_strut_side, *_strut_top, *_strut_rear, *_strut_3d,
            *_contact_dot_side, *_contact_dot_top, *_contact_dot_rear)

anim = FuncAnimation(fig_anim, _update, frames=len(anim_idx),
                     interval=40, blit=True)

# Display inline. constrained layout (set on the figure) handles spacing for the
# mixed 2-D/3-D axes; raise the embed limit so the full animation is kept (the
# embedded output is stripped on commit by the notebook pre-commit hook).
matplotlib.rcParams['animation.embed_limit'] = 64   # MB
HTML(anim.to_jshtml(fps=25))

In [ ]:
# ── Steady-state validation summary ───────────────────────────────────────────
# Check the last 2 seconds of simulation for steady-state values.

ss_mask = (t_s > T_END_S - 2.0) & wow
if ss_mask.sum() < 10:
    print('WARNING: fewer than 10 WoW samples in last 2 s — aircraft may not be settled.')
else:
    # Expected: contact force Fz (body) ≈ −W_N (body +Z = down, reaction is upward = negative)
    mean_Fz    = cf_force[ss_mask, 2].mean()
    mean_defl  = strut_defl[ss_mask].mean(axis=0)  # (3,)

    print('──── Steady-state validation (last 2 s on ground) ───────────────────')
    print(f'Aircraft weight W         = {W_N:.2f} N')
    print(f'Mean contact Fz body      = {mean_Fz:.2f} N  (expected ≈ {-W_N:.2f} N, i.e. upward)')
    print(f'Fz error                  = {mean_Fz - (-W_N):.3f} N  ({abs((mean_Fz + W_N)/W_N)*100:.1f}%)')
    print()
    print(f'Expected main-gear δ each = {delta_static*1000:.2f} mm  (W / (2·k_main))')
    for j, (name, d) in enumerate(zip(GEAR_NAMES, mean_defl)):
        print(f'  {name:8s}  mean δ = {d*1000:.2f} mm')

    print()
    mean_agl = agl_m[ss_mask].mean()
    print(f'Mean AGL (should be ≈ gear contact height): {mean_agl:.4f} m')

    # Flag potential bugs
    Fz_err_pct = abs((mean_Fz + W_N) / W_N) * 100
    if Fz_err_pct > 5.0:
        print(f'\n⚠  Contact Fz error {Fz_err_pct:.1f}% > 5% — possible gear spring or terrain-collision bug.')
    if mean_agl < -0.01:
        print(f'\n⚠  Negative AGL {mean_agl:.4f} m — terrain penetration: body collider or gear geometry bug.')
    if mean_agl > 0.30:
        print(f'\n⚠  AGL {mean_agl:.3f} m > 0.30 m at rest — aircraft not settling to ground: gear spring bug.')

## Touch-and-Go Validation

Same 6° approach as above, but after **5 s on the ground** the autopilot applies full throttle and
commands a 2 g pull-up.  Both the with-gear and body-collider-only scenarios are run.

Vertical markers: **dashed** = first touchdown, **solid** = go-around command issued.

In [ ]:
# ── Touch-and-go simulation ────────────────────────────────────────────────────
N_Z_ROTATE     = 2.0    # max commanded normal load factor during the go-around rotation
GAMMA_CLIMB_DEG = 6.0   # target climb flight-path angle for the go-around
T_SETTLE_S     = 0.5    # weight-on-wheels held this long before throttle (settled, not bouncing)
V_ROTATE_MPS   = 1.15 * V_STALL_MPS   # rotation speed: never rotate below this (no power-on stall)
T_END_TAG_S    = 60.0   # total duration (s) — enough to capture full cycle

def _run_touch_and_go(cfg, label=''):
    """Fly the stabilized FPA-hold approach, settle on the gear, full throttle, accelerate, then rotate gated on airspeed (not time)."""
    terrain_local = sim.FlatTerrain(TERRAIN_ELEV_M)
    ac_local = sim.Aircraft(json.dumps(cfg), dt_s=DT_S)
    ac_local.set_terrain(terrain_local)

    N_local = int(T_END_TAG_S / DT_S)
    r = dict(
        t_s       = np.zeros(N_local),
        agl_m     = np.zeros(N_local),
        alt_m     = np.zeros(N_local),
        vel_ned   = np.zeros((N_local, 3)),
        pitch_rad = np.zeros(N_local),
        alpha_rad = np.zeros(N_local),
        airspeed  = np.zeros(N_local),
        cf_force  = np.zeros((N_local, 3)),
        wow       = np.zeros(N_local, dtype=bool),
        throttle  = np.zeros(N_local),
        n_z_cmd   = np.zeros(N_local),
    )

    first_wow_t  = None
    settle_start = None
    powered      = False   # latched: full throttle, stays on through liftoff
    go_phase     = False   # latched: rotation, gated on airspeed
    go_t         = None    # time the go-around rotation was commanded
    gamma_target = abs(gamma_rad)
    gamma_climb  = math.radians(GAMMA_CLIMB_DEG)

    cmd_local = sim.AircraftCommand()
    cmd_local.n_y               = 0.0
    cmd_local.roll_rate_wind_rps = 0.0

    for i in range(N_local):
        s  = ac_local.state()
        cf = ac_local.contact_forces()
        t  = s.time_s
        v_h = math.hypot(s.velocity_north_mps, s.velocity_east_mps)

        if cf.weight_on_wheels and first_wow_t is None:
            first_wow_t = t
        # Settle detection: weight-on-wheels held continuously (not still bouncing).
        if cf.weight_on_wheels:
            if settle_start is None:
                settle_start = t
        else:
            settle_start = None
        # Sequenced touch-and-go (throttle and stick are NOT simultaneous): settle -> full throttle
        # (latched, stays on through liftoff) -> rotate once airspeed reaches V_ROTATE_MPS.
        if not powered and settle_start is not None and (t - settle_start) >= T_SETTLE_S:
            powered = True
        if powered and not go_phase and s.airspeed_m_s >= V_ROTATE_MPS:
            go_phase = True
            go_t = t

        cmd_local.throttle_nd = 1.0 if powered else 0.0
        if go_phase:
            # Rotate to and hold a climb flight-path angle (airspeed-gated, so never below stall).
            gamma_climb_cur = math.atan2(-s.velocity_down_mps, max(v_h, 0.1))   # climb-positive
            cmd_local.n_z = max(0.5, min(N_Z_ROTATE, 1.0 + 3.0 * (gamma_climb - gamma_climb_cur)))
        elif cf.weight_on_wheels:
            # On the ground, pre-rotation: hold (the settle term unloads the wing).
            cmd_local.n_z = 1.0
        else:
            # Airborne approach: flight-path-angle hold to the ground, no flare (same as _run_sim).
            gamma_cur = math.atan2(s.velocity_down_mps, max(v_h, 0.1))          # descent-positive
            cmd_local.n_z = max(0.2, min(1.8, math.cos(gamma_target) + FPA_GAIN * (gamma_cur - gamma_target)))

        r['t_s'][i]       = t
        r['agl_m'][i]     = ac_local.agl_m()
        r['alt_m'][i]     = s.altitude_m
        r['vel_ned'][i]   = [s.velocity_north_mps, s.velocity_east_mps, s.velocity_down_mps]
        r['pitch_rad'][i] = s.pitch_rad
        r['alpha_rad'][i] = s.alpha_rad
        r['airspeed'][i]  = s.airspeed_m_s
        r['cf_force'][i]  = cf.force_body_n
        r['wow'][i]       = cf.weight_on_wheels
        r['throttle'][i]  = cmd_local.throttle_nd
        r['n_z_cmd'][i]   = cmd_local.n_z

        ac_local.step(cmd_local, dt_s=DT_S, rho_kgm3=RHO_KGM3)

    r['td_t']  = first_wow_t
    r['go_t']  = go_t
    r['label'] = label
    return r


results_tag_gear = _run_touch_and_go(approach_cfg, label='Touch-and-go (gear)')

_cfg_tag_no_gear = copy.deepcopy(approach_cfg)
del _cfg_tag_no_gear['landing_gear']
results_tag_no_gear = _run_touch_and_go(_cfg_tag_no_gear, label='Touch-and-go (body collider only)')

for rr in [results_tag_gear, results_tag_no_gear]:
    td, go = rr['td_t'], rr['go_t']
    wow_times = rr['t_s'][rr['wow']]
    last_wow  = wow_times[-1] if wow_times.size else None
    if td is None:
        print(f'{rr["label"]}: no touchdown detected')
    else:
        lw_str = f'{last_wow:.3f} s' if last_wow is not None else 'none'
        td_str = f'{td:.3f} s' if td is not None else 'none'
        go_str = f'{go:.3f} s' if go is not None else 'none (never settled / below V_rotate)'
        print(f'{rr["label"]}: T/D = {td_str} | go-around = {go_str} | last WoW = {lw_str}')

In [ ]:
# ── Touch-and-go comparison plots ─────────────────────────────────────────────
_TAG_COLORS = {
    'Touch-and-go (gear)':               'tab:blue',
    'Touch-and-go (body collider only)': 'tab:orange',
}

fig_tag, axes_tag = plt.subplots(3, 2, figsize=(13, 11))
fig_tag.suptitle('Touch-and-Go — Gear vs Body Collider Only\n'
                 '(dashed = first T/D, solid = go-around command)',
                 fontsize=12, fontweight='bold')

for rr in [results_tag_gear, results_tag_no_gear]:
    col  = _TAG_COLORS[rr['label']]
    lbl  = rr['label']
    t    = rr['t_s']
    td_t = rr['td_t']
    go_t = rr['go_t']

    def _vlines(ax):
        if td_t: ax.axvline(td_t, color=col, lw=0.9, ls='--', alpha=0.6)
        if go_t: ax.axvline(go_t, color=col, lw=1.3, ls='-',  alpha=0.8)

    # AGL
    ax = axes_tag[0, 0]
    ax.plot(t, rr['agl_m'], color=col, label=lbl)
    _vlines(ax)

    # Airspeed
    ax = axes_tag[0, 1]
    ax.plot(t, rr['airspeed'], color=col, label=lbl)
    _vlines(ax)

    # Pitch
    ax = axes_tag[1, 0]
    ax.plot(t, np.degrees(rr['pitch_rad']), color=col, label=lbl)
    _vlines(ax)

    # WoW
    ax = axes_tag[1, 1]
    ax.plot(t, rr['wow'].astype(float), color=col, label=lbl, lw=1.2)
    _vlines(ax)

    # Contact normal force
    ax = axes_tag[2, 0]
    ax.plot(t, -rr['cf_force'][:, 2], color=col, label=lbl)
    _vlines(ax)

    # Throttle and n_z command
    ax = axes_tag[2, 1]
    ax.plot(t, rr['throttle'], color=col, lw=1.5, label=f'{lbl} — throttle')
    ax.plot(t, rr['n_z_cmd'],  color=col, lw=1.0, ls='--', label=f'{lbl} — n_z cmd')
    _vlines(ax)

# Decorate ────────────────────────────────────────────────────────────────────
ax = axes_tag[0, 0]
ax.axhline(0.0, color='saddlebrown', lw=1.5)
ax.set_xlabel('Time (s)'); ax.set_ylabel('AGL (m)')
ax.set_title('Height Above Ground (CG)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes_tag[0, 1]
ax.set_xlabel('Time (s)'); ax.set_ylabel('Airspeed (m/s)')
ax.set_title('Airspeed'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes_tag[1, 0]
ax.set_xlabel('Time (s)'); ax.set_ylabel('Pitch (°)')
ax.set_title('Pitch Angle'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes_tag[1, 1]
ax.set_ylim(-0.1, 1.3)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Weight-on-Wheels')
ax.set_title('Weight-on-Wheels Flag'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes_tag[2, 0]
ax.axhline(W_N, color='k', lw=0.8, ls='--', label=f'Weight {W_N:.1f} N')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Force (N)')
ax.set_title('Contact Normal Force (upward = −Fz_body)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes_tag[2, 1]
ax.set_xlabel('Time (s)'); ax.set_ylabel('Command value')
ax.set_title('Throttle (solid) and n_z Command (dashed)'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

fig_tag.tight_layout()
plt.show()